# Day 4: More Operations — Expanding the Vocabulary

**Learning Objective**: Implement additional operations (power, division, negation, subtraction) with correct backward passes.

Today we'll add `__pow__`, `__neg__`, `__sub__`, `__truediv__` and their reverse operations to complete our autograd engine!

In [ ]:
import math
from graphviz import Digraph

## Theory: New Derivative Rules

### Power Rule
$$\frac{d(x^n)}{dx} = n \cdot x^{n-1}$$

### Negation
$$\frac{d(-x)}{dx} = -1$$

### Subtraction (via negation)
$$a - b = a + (-b)$$

### Division (via power)
$$\frac{a}{b} = a \cdot b^{-1}$$

**Key insight**: We can build subtraction and division from existing operations!

## Value Class with All Operations

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward
        return out

    def __pow__(self, other):
        """Power operation with power rule backward pass."""
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            # Power rule: d(x^n)/dx = n * x^(n-1)
            self.grad += (other * self.data ** (other - 1)) * out.grad
        
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            self.grad += (1 - t**2) * out.grad
        
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')
        
        def _backward():
            self.grad += out.data * out.grad
        
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    # === NEW OPERATIONS ===
    
    def __neg__(self):  # -self
        """Negation: implemented as multiplication by -1."""
        return self * -1

    def __radd__(self, other):  # other + self
        return self + other

    def __sub__(self, other):  # self - other
        """Subtraction: implemented as self + (-other)."""
        return self + (-other)

    def __rsub__(self, other):  # other - self
        return Value(other) - self

    def __rmul__(self, other):  # other * self
        return self * other

    def __truediv__(self, other):  # self / other
        """Division: implemented as self * other^(-1)."""
        return self * other**-1

    def __rtruediv__(self, other):  # other / self
        return Value(other) * self**-1

## Visualization Helper

In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

## Test 1: Power Operation

Testing `a ** 3` where `a = 2.0`

Expected gradient: `3 * 2^2 = 12.0`

In [ ]:
a = Value(2.0, label='a')
b = a ** 3; b.label = 'b'

b.backward()

print(f"b.data = {b.data:.4f} (should be 8.0)")
print(f"a.grad = {a.grad:.4f} (should be 12.0)")

assert abs(b.data - 8.0) < 1e-5, "Power forward failed!"
assert abs(a.grad - 12.0) < 1e-5, "Power backward failed!"
print("✅ Power operation works!")

## Test 2: Negation

Testing `-a` where `a = 3.0`

In [ ]:
a = Value(3.0, label='a')
b = -a; b.label = 'b'

b.backward()

print(f"b.data = {b.data:.4f} (should be -3.0)")
print(f"a.grad = {a.grad:.4f} (should be -1.0)")

assert abs(b.data - (-3.0)) < 1e-5, "Negation forward failed!"
assert abs(a.grad - (-1.0)) < 1e-5, "Negation backward failed!"
print("✅ Negation works!")

## Test 3: Subtraction

Testing `a - b` where `a = 5.0`, `b = 3.0`

Expected: `∂L/∂a = 1.0`, `∂L/∂b = -1.0`

In [ ]:
a = Value(5.0, label='a')
b = Value(3.0, label='b')
c = a - b; c.label = 'c'

c.backward()

print(f"c.data = {c.data:.4f} (should be 2.0)")
print(f"a.grad = {a.grad:.4f} (should be 1.0)")
print(f"b.grad = {b.grad:.4f} (should be -1.0)")

assert abs(c.data - 2.0) < 1e-5, "Subtraction forward failed!"
assert abs(a.grad - 1.0) < 1e-5, "Subtraction a.grad failed!"
assert abs(b.grad - (-1.0)) < 1e-5, "Subtraction b.grad failed!"
print("✅ Subtraction works!")

## Test 4: Division

Testing `a / b` where `a = 6.0`, `b = 2.0`

Expected gradients:
- `∂(a/b)/∂a = 1/b = 0.5`
- `∂(a/b)/∂b = -a/b² = -6/4 = -1.5`

In [ ]:
a = Value(6.0, label='a')
b = Value(2.0, label='b')
c = a / b; c.label = 'c'

c.backward()

print(f"c.data = {c.data:.4f} (should be 3.0)")
print(f"a.grad = {a.grad:.4f} (should be 0.5)")
print(f"b.grad = {b.grad:.4f} (should be -1.5)")

assert abs(c.data - 3.0) < 1e-5, "Division forward failed!"
assert abs(a.grad - 0.5) < 1e-5, "Division a.grad failed!"
assert abs(b.grad - (-1.5)) < 1e-5, "Division b.grad failed!"
print("✅ Division works!")

## Test 5: Reverse Operations

Testing `5 - a` and `12 / a`

In [ ]:
a = Value(3.0, label='a')

# Test rsub: 5 - a
b = 5 - a; b.label = 'b'
print(f"5 - a = {b.data:.4f} (should be 2.0)")
assert abs(b.data - 2.0) < 1e-5

# Test rtruediv: 12 / a
c = 12 / a; c.label = 'c'
print(f"12 / a = {c.data:.4f} (should be 4.0)")
assert abs(c.data - 4.0) < 1e-5

print("✅ Reverse operations work!")

## Test 6: Complex Expression

Testing: $L = \frac{(a - b)^2}{c}$

With `a=5`, `b=3`, `c=2`:
- $L = (5-3)^2 / 2 = 4/2 = 2$
- $\frac{\partial L}{\partial a} = \frac{2(a-b)}{c} = \frac{2 \cdot 2}{2} = 2$
- $\frac{\partial L}{\partial b} = \frac{-2(a-b)}{c} = \frac{-2 \cdot 2}{2} = -2$
- $\frac{\partial L}{\partial c} = \frac{-(a-b)^2}{c^2} = \frac{-4}{4} = -1$

In [ ]:
a = Value(5.0, label='a')
b = Value(3.0, label='b')
c = Value(2.0, label='c')

# L = (a - b)^2 / c
diff = a - b; diff.label = 'a-b'
sq = diff ** 2; sq.label = '(a-b)²'
L = sq / c; L.label = 'L'

L.backward()

print(f"L.data = {L.data:.4f} (should be 2.0)")
print(f"a.grad = {a.grad:.4f} (should be 2.0)")
print(f"b.grad = {b.grad:.4f} (should be -2.0)")
print(f"c.grad = {c.grad:.4f} (should be -1.0)")

assert abs(L.data - 2.0) < 1e-5
assert abs(a.grad - 2.0) < 1e-5
assert abs(b.grad - (-2.0)) < 1e-5
assert abs(c.grad - (-1.0)) < 1e-5

print("\n🎉 All gradients correct!")

In [ ]:
# Visualize the computation graph
draw_dot(L)

## Summary

Today we implemented:

1. **`__pow__`** with power rule: $\frac{d(x^n)}{dx} = n \cdot x^{n-1}$
2. **`__neg__`** as multiplication by -1
3. **`__sub__`** using addition and negation: $a - b = a + (-b)$
4. **`__truediv__`** using power: $a / b = a \cdot b^{-1}$
5. **Reverse operations** (`__rsub__`, `__rtruediv__`) for expressions like `5 - a`

**Key insight**: We built complex operations from simpler ones, letting the chain rule handle gradient computation automatically!

---

*Previous: [Day 3 — Chain Rule](./day_03_chain_rule.ipynb)*  
*Next: Day 5 — Neurons*